# NTP / temporal calibration — one run

Characterizes clock synchronization between the agents for a single recorded run.
Input is the folder written by `extract_bag.py` (per-topic Parquet tables plus `stamp_audit.parquet`).

Sections:
1. Configuration
2. Load
3. Who is server, who are clients
4. Per-client offset statistics and clock steps
5. NTP events
6. NTP-independent check from header stamps
7. Figure
8. Paper text

In [ ]:
from pathlib import Path
import glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- configuration -----------------------------------------------------------
RUN       = "coop2"
EXTRACTS  = Path("extracts") / RUN          # output folder of extract_bag.py
OUT       = Path("results") / RUN / "ntp"   # where tables, figure and LaTeX go
STEP_THRESHOLD_MS = 1.0                     # |offset_delta| above this counts as a clock step
SENSOR_PERIOD_MS  = None                    # None = take the shortest median period found in the stamp audit

OUT.mkdir(parents=True, exist_ok=True)

AGENT_COLOR = {"mobile_1": "#2a78d6", "mobile_2": "#eb6834", "infra_1": "#1baf7a"}
STEP_COLOR, EVENT_COLOR, TEXT, TEXT2, GRID = "#e34948", "#eda100", "#0b0b0b", "#52514e", "#e6e5e1"
color_for = lambda node: AGENT_COLOR.get(node, "#4a3aa7")
node_of   = lambda topic: topic.strip("/").split("/")[0]

def ecdf(x):
    x = np.sort(np.asarray(x, dtype=float)); x = x[~np.isnan(x)]
    return x, np.arange(1, len(x) + 1) / max(len(x), 1)

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

## 2. Load

In [ ]:
def load_glob(pattern):
    frames = []
    for f in sorted(glob.glob(str(EXTRACTS / pattern))):
        df = pd.read_parquet(f); df["topic"] = "/" + Path(f).stem.replace("__", "/"); frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else None

ntp    = load_glob("*ntp__status.parquet")
events = load_glob("*ntp__events.parquet")
audit  = pd.read_parquet(EXTRACTS / "stamp_audit.parquet") if (EXTRACTS / "stamp_audit.parquet").exists() else None
assert ntp is not None, f"no *ntp__status.parquet under {EXTRACTS}"

t0_ns = int(ntp["log_time_ns"].min())
if audit is not None and len(audit):
    t0_ns = min(t0_ns, int(audit["log_time_ns"].min()))

ntp["t_s"]            = (ntp["log_time_ns"] - t0_ns) / 1e9
ntp["node"]           = ntp["topic"].map(node_of)
ntp["offset_ms"]      = ntp["offset_seconds"] * 1e3
ntp["delay_ms"]       = ntp["delay_seconds"] * 1e3
ntp["jitter_ms"]      = ntp["jitter_seconds"] * 1e3
ntp["offset_delta_ms"]= ntp["offset_delta_seconds"] * 1e3
ntp["clock_stepped"]  = ntp["clock_stepped"].astype(bool)
print(f"{len(ntp)} NTP status rows from {ntp['topic'].nunique()} topics; "
      f"{0 if events is None else len(events)} event rows; "
      f"audit: {0 if audit is None else len(audit)} rows")
ntp.head(3).T

## 3. Who is server, who are clients

`sync_source` names the server each client follows. `mobile_1` publishes no NTP topic in this bag, so it should appear here as the sync source of the others.

In [ ]:
roles = (ntp.groupby(["topic", "role", "hostname", "sync_source"], dropna=False)
           .agg(n=("seq", "size"),
                stratum=("stratum", lambda s: int(s.mode().iloc[0])),
                synchronized_frac=("synchronized", "mean"),
                rate_hz=("t_s", lambda t: (len(t) - 1) / max(t.max() - t.min(), 1e-9)),
                t_first_s=("t_s", "min"), t_last_s=("t_s", "max"))
           .reset_index())
roles.to_csv(OUT / "ntp_roles.csv", index=False)
roles

## 4. Per-client offset statistics and clock steps

Steps are counted two ways: the monitor's own `clock_stepped` flag, and any `offset_delta` larger than `STEP_THRESHOLD_MS`.

In [ ]:
rows, series = [], {}
for (topic, role, host), g in ntp.groupby(["topic", "role", "hostname"]):
    g = g.sort_values("t_s"); key = host if role == "client" else f"{host} ({role})"; series[key] = g
    abs_off = g["offset_ms"].abs()
    steps_flag  = int((g["clock_stepped"] & ~g["clock_stepped"].shift(1, fill_value=False)).sum())
    steps_delta = int((g["offset_delta_ms"].abs() > STEP_THRESHOLD_MS).sum())
    warn = sorted({w for ws in g["warnings"] for w in (list(ws) if ws is not None else [])})
    rows.append(dict(run=RUN, topic=topic, node=node_of(topic), role=role, hostname=host,
        sync_source=g["sync_source"].mode().iloc[0], stratum=int(g["stratum"].mode().iloc[0]), n=len(g),
        duration_s=float(g["t_s"].max() - g["t_s"].min()),
        offset_mean_ms=g["offset_ms"].mean(), offset_median_ms=g["offset_ms"].median(), offset_std_ms=g["offset_ms"].std(),
        abs_offset_mean_ms=abs_off.mean(), abs_offset_p95_ms=abs_off.quantile(0.95), abs_offset_max_ms=abs_off.max(),
        t_of_max_abs_offset_s=g.loc[abs_off.idxmax(), "t_s"],
        delay_median_ms=g["delay_ms"].median(), delay_max_ms=g["delay_ms"].max(), jitter_median_ms=g["jitter_ms"].median(),
        root_dispersion_median_ms=g["root_dispersion"].median() * 1e3, freq_error_mean_ppm=g["frequency_error_ppm"].mean(),
        poll_interval_mode_s=int(g["poll_interval_seconds"].mode().iloc[0]),
        reach_min=int(g["reach_register"].min()), reachability_min_pct=int(g["reachability_percent"].min()),
        synchronized_frac=g["synchronized"].astype(bool).mean(),
        clock_steps_flagged=steps_flag, clock_steps_by_delta=steps_delta,
        step_times_s=json.dumps([round(float(x), 2) for x in g.loc[g["clock_stepped"], "t_s"]][:20]),
        warnings="; ".join(warn)))
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "ntp_summary.csv", index=False)
SHOW = ["hostname", "role", "sync_source", "stratum", "n", "offset_mean_ms", "offset_median_ms", "abs_offset_p95_ms",
        "abs_offset_max_ms", "delay_median_ms", "jitter_median_ms", "poll_interval_mode_s", "reach_min",
        "clock_steps_flagged", "clock_steps_by_delta"]
summary[SHOW].round(3)

## 5. NTP events

In [ ]:
if events is not None and len(events):
    events["t_s"] = (events["log_time_ns"] - t0_ns) / 1e9
    display(events[["t_s", "topic", "data"]])
else:
    print("no NTP event messages in this run")

## 6. NTP-independent check from header stamps

Every sensor is stamped in software on arrival at its host, so header stamp minus recorder log time is transport latency plus the clock offset between that node and the recorder. In the recorder's own clock it is slightly negative (pure latency). A node whose topics sit systematically off the recorder's has a clock offset of that size, independent of what the NTP monitor reports. The shortest median period among the topics is the sync bound the offsets are compared against.

In [ ]:
audit_df, sensor_period_ms, fastest = None, SENSOR_PERIOD_MS, "(user supplied)"
if audit is not None and len(audit):
    audit = audit.dropna(subset=["header_stamp_ns"]).copy()
    audit["t_s"] = (audit["log_time_ns"] - t0_ns) / 1e9
    arows = []
    for (node, topic), g in audit.groupby(["node", "topic"]):
        g = g.sort_values("header_stamp_ns"); dt = np.diff(g["header_stamp_ns"].to_numpy()) / 1e6
        d = g["stamp_minus_log_ms"]
        arows.append(dict(node=node, topic=topic, type=g["type"].iloc[0], n=len(g),
            period_median_ms=float(np.median(dt)) if len(dt) > 10 else np.nan,
            stamp_minus_log_median_ms=d.median(), stamp_minus_log_p05_ms=d.quantile(0.05),
            stamp_minus_log_p95_ms=d.quantile(0.95), stamp_minus_log_iqr_ms=d.quantile(0.75) - d.quantile(0.25)))
    audit_df = pd.DataFrame(arows).sort_values(["node", "topic"])
    audit_df.to_csv(OUT / "ntp_audit.csv", index=False)
    valid = audit_df[(audit_df["n"] >= 100) & audit_df["period_median_ms"].notna()]
    if SENSOR_PERIOD_MS is None and len(valid):
        sensor_period_ms = float(valid["period_median_ms"].min())
        fastest = valid.loc[valid["period_median_ms"].idxmin(), "topic"]
    print("per-node median of the per-topic medians (ms):")
    display(audit_df.groupby("node")["stamp_minus_log_median_ms"].median().round(3).to_frame())
    display(audit_df.round(3))
print(f"shortest sensor period: {sensor_period_ms} ms ({fastest})")

## 7. Figure

(a) client clock offset over time, clock steps in red, NTP events in yellow. (b) ECDF of |offset| per client with the shortest sensor period marked. (c) ECDF of header stamp minus log time, one line per topic, colored by node.

In [ ]:
plt.rcParams.update({"font.size": 8, "axes.edgecolor": GRID, "axes.labelcolor": TEXT,
                     "xtick.color": TEXT2, "ytick.color": TEXT2, "text.color": TEXT})
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.4), constrained_layout=True)

ax = axes[0]
for key, g in series.items():
    ax.plot(g["t_s"], g["offset_ms"], lw=1.2, color=color_for(g["node"].iloc[0]), label=key)
    for ts in g.loc[g["clock_stepped"], "t_s"]:
        ax.axvline(ts, color=STEP_COLOR, lw=0.8, alpha=0.8)
if events is not None and len(events):
    for t in events["t_s"]:
        ax.axvspan(t - 0.4, t + 0.4, color=EVENT_COLOR, alpha=0.35, lw=0)
ax.axhline(0, color=GRID, lw=0.8); ax.set_xlabel("time in run [s]"); ax.set_ylabel("NTP offset to server [ms]")
ax.set_title("(a) client clock offset", loc="left", fontsize=8); ax.legend(frameon=False, fontsize=7); ax.grid(True, color=GRID, lw=0.5)

ax = axes[1]
for key, g in series.items():
    x, y = ecdf(g["offset_ms"].abs()); ax.plot(x, y, lw=1.5, color=color_for(g["node"].iloc[0]), label=key)
if sensor_period_ms:
    ax.axvline(sensor_period_ms, color=TEXT2, lw=0.8, ls="--")
    ax.text(sensor_period_ms, 0.05, f" shortest sensor\n period {sensor_period_ms:.1f} ms", fontsize=6.5, color=TEXT2, va="bottom")
ax.set_xscale("log"); ax.set_xlabel("|offset| [ms]"); ax.set_ylabel("ECDF")
ax.set_title("(b) offset distribution", loc="left", fontsize=8); ax.grid(True, color=GRID, lw=0.5, which="both")

ax = axes[2]
if audit is not None and len(audit):
    seen = set()
    for (node, topic), g in audit.groupby(["node", "topic"]):
        if len(g) < 50: continue
        x, y = ecdf(g["stamp_minus_log_ms"])
        ax.plot(x, y, lw=0.9, color=color_for(node), label=node if node not in seen else None); seen.add(node)
    ax.set_xlabel("header stamp − log time [ms]"); ax.set_ylabel("ECDF (one line per topic)"); ax.legend(frameon=False, fontsize=7)
ax.set_title("(c) NTP-independent check", loc="left", fontsize=8); ax.grid(True, color=GRID, lw=0.5)

fig.savefig(OUT / "fig_ntp.pdf"); fig.savefig(OUT / "fig_ntp.png", dpi=200)
plt.show()

## 8. Paper text

In [ ]:
tt = lambda s: "\\texttt{" + str(s).replace("_", "\\_") + "}"
clients = summary[summary["role"].str.contains("client")]
server  = clients["sync_source"].mode().iloc[0] if len(clients) else "?"
strata  = ", ".join(str(s) for s in sorted(clients["stratum"].unique()))
rate    = ", ".join(f"{r:.1f}" for r in roles["rate_hz"])
parts = [f"{tt(r.hostname)} had a mean offset of {r.offset_mean_ms:.2f}\\,ms (mean $|\\cdot|$ {r.abs_offset_mean_ms:.2f}\\,ms, "
         f"95th percentile {r.abs_offset_p95_ms:.2f}\\,ms, maximum {r.abs_offset_max_ms:.2f}\\,ms) with a median round-trip delay "
         f"of {r.delay_median_ms:.2f}\\,ms and {int(r.clock_steps_flagged)} clock step{'s' if r.clock_steps_flagged != 1 else ''}"
         for r in clients.itertuples()]
max_all = clients["abs_offset_max_ms"].max()
audit_sentence = bound_sentence = ""
if audit_df is not None:
    spread = audit_df.groupby("node")["stamp_minus_log_median_ms"].median()
    audit_sentence = (f" As an NTP-independent check, the per-node median of header stamp minus recorder receive time spans "
                      f"{spread.min():.2f} to {spread.max():.2f}\\,ms across nodes, bounding clock offset plus transport latency.")
if sensor_period_ms:
    bound_sentence = (f" All offsets are below the shortest sensor period in the recording ({sensor_period_ms:.1f}\\,ms), "
                      f"so cross-agent messages can be associated by timestamp without further alignment."
                      if max_all < sensor_period_ms else
                      f" The maximum offset exceeds the shortest sensor period ({sensor_period_ms:.1f}\\,ms); "
                      f"cross-agent association of the fastest topics needs the recorded offsets applied.")
tex = f"""\\subsubsection{{NTP Synchronization}}
All agents are synchronized over NTP on the shared wireless network.
{tt(server)} acts as the NTP server and the other agents synchronize to it as
stratum-{strata} clients; each client publishes its NTP state at about {rate}\\,Hz throughout every run.
Over run {tt(RUN)}, {'; '.join(parts)}.{audit_sentence}{bound_sentence}
No sensor is hardware-triggered or hardware-timestamped: every message is stamped in software by
its driver on arrival at the host, so the header stamps carry the NTP-aligned host clock plus the
driver's arrival latency, and the offsets above bound clock disagreement between agents, not
sensor exposure time.
"""
(OUT / "ntp_subsection.tex").write_text(tex)
print(tex)